# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ailya-Shah/INTERNSHIP-TASKS/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Primarily binary classification, used for ranking.** The lane guide's task-type table maps "which ones first?" questions to ranking/scoring and "will this one X?" questions (with an observed label) to classification -- my question is really both: I'm predicting a yes/no outcome (is this page on page one?) but the point of predicting it is to *rank* pages by how likely they are to be page-one-worthy, so an editor knows which to review first.

So concretely: I'll train a classifier (logistic regression, then random forest) that outputs a probability per page, then evaluate and use that probability as a ranking score (Precision@K), not just as a label. The signal-analysis value -- which features carry the ranking, and in which direction -- comes from the model's coefficients/importances, which is the actual deliverable for this lane.

In [1]:
# No query needed for this section -- reasoning only.
print("Task type: binary classification, evaluated and used as a ranking score.")


Task type: binary classification, evaluated and used as a ranking score.


## 2. Target or proxy

**Target: `is_page_one`** -- a page is labeled 1 if its impression-weighted average search position falls between 1 and 10, else 0.

**Where the label comes from:** `avg_position` (starter data) / `gsc_avg_position` (warehouse) is an **observed measurement** from Google Search Console -- a real recorded outcome, not a hand-authored proxy like a product `health_score` or `priority_tag`. That matters: per the framing skill, "the target must be observed, not defined" -- a label built from someone's existing rule just teaches the model to reproduce that rule. My threshold (1-10 = page one) is a simple, transparent *cutoff* I'm choosing on a real observed number, which is a different and safer thing than inheriting a pre-computed decision flag.

One thing I'm being careful about: rows where `avg_position == 0` mean "no position data," not literal rank zero (confirmed in w01) -- I exclude those rather than let them corrupt the label.

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
valid = df[df["avg_position"] > 0].copy()
valid["is_page_one"] = valid["avg_position"].between(1, 10)

print(f"labeled rows: {len(valid):,} (dropped {len(df) - len(valid):,} with no position data)")
print("label source column: avg_position (observed GSC measurement, not a product flag)")
print(f"base rate: {valid['is_page_one'].mean():.1%}")


labeled rows: 28,795 (dropped 1,205 with no position data)
label source column: avg_position (observed GSC measurement, not a product flag)
base rate: 44.8%


## 3. Success metric

**Primary metric: Average Precision (PR-AUC), with Precision@K reported alongside it.** Because I'm ranking pages for a human to review in priority order, precision at the top of the list matters more than overall accuracy across all 30,000 pages -- Average Precision captures that ranking quality, and Precision@50 gives a very concrete number ("of the top 50 pages my model flags, how many are actually page-one-caliber?").

**What "good" means:** the model must beat two floors on the same held-out-client split: (1) a `DummyClassifier` majority-class floor, and (2) a transparent single-signal baseline (e.g. ranking by `search_volume` alone). I will always report the **base rate** next to every number, since a bare Precision@50 of 0.60 means nothing until you know whether guessing already gets 0.55 or 0.10.

Secondary: ROC AUC, mainly as a familiar sanity-check number alongside Average Precision, not as the headline.

In [3]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
valid = df[df["avg_position"] > 0].copy()
valid["is_page_one"] = valid["avg_position"].between(1, 10).astype(int)

# floor-below-the-floor: what a majority-class guess alone gets you
X = valid[["word_count"]].fillna(0)
y = valid["is_page_one"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print(f"base rate (test): {y_te.mean():.3f}")
print(f"dummy accuracy:   {dummy.score(X_te, y_te):.3f}  <- the number any real model must clear")


base rate (test): 0.448
dummy accuracy:   0.552  <- the number any real model must clear


## 4. The unit of analysis, as a real dataframe

**One row = one content page.** Each page is uniquely identified by `content_id`; a page's row aggregates its own search/traffic metrics over a fixed window (trailing 90 days in the starter data; my capstone window in the warehouse). It is **not** one row per day, per client, or per query -- those are separate tables (the daily fact and the query-mix table) that I aggregate *down* to this one-row-per-page grain before modeling.

In [4]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# confirm the grain: content_id should be unique (one row per page)
dupes = df["content_id"].duplicated().sum()
print(f"rows: {len(df):,} | unique content_id: {df['content_id'].nunique():,} | duplicate content_ids: {dupes}")

df[["content_id", "client_id", "word_count", "avg_position", "content_type", "main_intent"]].head()


rows: 30,000 | unique content_id: 30,000 | duplicate content_ids: 0


,content_id,client_id,word_count,avg_position,content_type,main_intent
0,content_304f48230142,client_f369cb89fc,3221.0,10.6,keyword article,transactional
1,content_a1fb4e703a9e,client_4e07408562,2481.0,20.3,keyword article,informational
2,content_9aa793d4d895,client_7f2253d7e2,3515.0,36.5,keyword article,informational
3,content_331d6c4de07b,client_19581e27de,NaN,6.2,keyword article,commercial
4,content_d99b7a2d90ca,client_3fdba35f04,2803.0,44.0,keyword article,informational


## 5. Why ML beats a fixed rule here

A fixed rule would need a human to pre-decide which single signal (or hand-tuned combination) predicts page-one status, and to get the thresholds right by eye. My own w01 check already shows that the obvious naive rule -- "longer content ranks better" -- doesn't even hold in the right direction here (page-one pages had a *lower* median word count, not higher). If the single most intuitive signal is already backwards, a hand-written if-statement built on intuition is exactly the wrong tool.

What's really happening is that page-one status plausibly depends on **several signals interacting at once** -- length, freshness, demand (`search_volume`, `competition`), and query concentration -- in ways that aren't simply additive or the same direction for every content type. That's precisely the situation the framing skill flags as ML's actual use case: "the pattern is real but too messy to write by hand -- many signals, tangled, shifting." A simple learned model (logistic regression first, for readable coefficients) can weigh and combine those signals the way a rule-of-thumb can't, while still staying interpretable enough that I can report *which* signals mattered and in *which* direction.

Tying this to action: the ranked output feeds directly into the editor workflow from w01 — the model's probability score becomes the priority order for review, and permutation importance/coefficients tell the editor why a page was flagged (thin content, stale update, low query focus), so the action is never a black-box score alone but a reason-coded recommendation.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.